# Week 07 · Tool use — hardened multi-turn loop (proposed fix)

Proposed rewrite of `week07_tools.ipynb`. Your original is untouched. Each section below says **what changed and why**.

**Why the old loop never stopped**

| # | Cause | Fix in this notebook |
|---|---|---|
| 1 | `tool_choice={"type": "tool", ...}` sent on **every** turn means Claude *must* call a tool again, so `end_turn` can never happen | `first_tool_choice` is used on turn 1 only |
| 2 | `batch_tool`'s `except` branches read `inv.name`, but `inv` is a dict, so any inner failure became `'dict' object has no attribute 'name'`. Claude got an error it could not act on and retried | Every call returns `(ok, value)` and never raises. Errors tell Claude what to send instead |
| 3 | One failed inner call raised for the **whole** batch (`is_error=True`), including calls that had already run, like `set_reminder`. Claude retried them all, so reminders could be duplicated | Results are reported per call, and `set_reminder` is idempotent |
| 4 | No turn limit | `MAX_TURNS`, a repeat-failure circuit breaker, and a final wrap-up call with tools disabled |

**Can Claude read several tool outputs at once?** Yes. Your `run_tools` already does it correctly: one `tool_result` per `tool_use`, all in **one** user message. Current models make parallel calls on their own, so `batch_tool` is now optional (`ENABLE_BATCH_TOOL`).

---
## IMPORTS AND SETTINGS
---

Line 'INFO httpx2: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"' comes from httpx2, the HTTP library the SDK uses to send requests. That library writes one INFO line per request.
The line is useful in one way: each line is one model call, so it matches your turn count. A 200 OK means the request succeeded.

logging.basicConfig(level=logging.INFO, **force=True**) set every logger to INFO, including httpx2's.

The anthropic logger now shows its INFO messages too, but it has only one: a retry notice. Its default level was never DEBUG, though; it simply follows Python's root logger (WARNING by default).

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import time
from collections import Counter
from collections.abc import Callable
from dataclasses import KW_ONLY, dataclass
from datetime import UTC, datetime, timedelta
from inspect import signature
from typing import Annotated, Any, Final, Literal, NotRequired, Required, TypedDict, cast
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from anthropic.types import (
    Message,
    MessageParam,
    TextBlock,
    ToolChoiceParam,
    ToolParam,
    ToolResultBlockParam,
    ToolUseBlock,
)
from pydantic import BaseModel, ConfigDict, Json, StringConstraints, ValidationError, validate_call
from rich.console import Console
from rich.markdown import Markdown

from learning_journey.claude.client import get_client

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("httpx2").setLevel(logging.WARNING)  # hide httpx2's INFO logs
logger = logging.getLogger("week07_tools")
console = Console()

# --- Loop and tool settings, in one place ---
DEFAULT_TZ: Final[str] = "America/New_york"  # the USER's IANA zone, not the server's clock
MAX_TURNS: Final[int] = 8  # hard stop: at most this many model calls (+1 wrap-up call)
MAX_SAME_FAILURES: int = 2  # circuit breaker: block a call that already failed this often
ENABLE_BATCH_TOOL: Final[bool] = False  # native parallel tool use makes batch_tool optional

In [ ]:
class LLMParams(TypedDict, total=False):
    model: Required[str]
    max_tokens: Required[int]
    messages: Required[list[MessageParam]]
    system: NotRequired[str]
    stop_sequences: NotRequired[list[str]]
    tools: NotRequired[list[ToolParam]]
    tool_choice: NotRequired[ToolChoiceParam]
    # ^ was ToolChoiceToolParam. The union also allows {"type": "none"},
    #   which the loop uses for its final wrap-up call.


@dataclass(frozen=True, slots=True)
class ToolsInfo:
    func: Callable[..., Any]
    _: KW_ONLY
    name: str
    schema: ToolParam
    side_effects: bool = False
    # ^ True for tools that change something (set_reminder). A retry of one of these
    #   can repeat the change, so they must be idempotent: same input -> no duplicate.


class ToolError(Exception):
    """An expected tool failure. The message goes to Claude word for word, so write it
    as an instruction: what was wrong and what to send instead."""

---
## BATCH MODEL (Pydantic v2)
---
Same idea as before. `extra="forbid"` makes an unknown key an error instead of being silently dropped.

In [ ]:
# ---------------------------------------------------------------------
# 1. MODELS: mirror batch_tool_schema["input_schema"] one-to-one
# ---------------------------------------------------------------------
class Invocation(BaseModel):
    """One inner tool call inside the batch (one item of the array)."""

    model_config = ConfigDict(extra="forbid")

    name: str
    # ^ Tool to run, e.g. "get_weather". Must be a key in TOOL_REGISTRY.

    arguments: Json[dict[str, Any]]
    # ^ Claude sends this as a JSON *string*: '{"location": "SF"}'
    #   Json[...] does two steps for you:
    #     1) json.loads() the string       ""   -> fails if not valid JSON
    #     2) validate the result as dict[...] -> fails if it's a list/number
    #   After validation, inv.arguments is a real dict: {"location": "SF"}
    #   Note: a dict passed directly (not a string) FAILS. Json[] only
    #   accepts str / bytes / bytearray.

---
## REGISTER TOOL FUNCTIONS
---
Two guards are added at registration time:

1. **Drift guard.** The JSON schema Claude sees and the Python signature must list the same parameters. If they don't, the cell fails when you run it, not halfway through a conversation.
2. **`validate_call`.** Pydantic checks Claude's arguments against your type hints *before* the function body runs. `duration=1.5` for an `int`, or `unit="fortnights"` for a `Literal`, raises a `ValidationError` that names the exact field, and that text is what Claude sees.

`strict: true` on each schema is the model-side half of the same guarantee: grammar-constrained sampling makes Claude's input match the schema. Strict mode does not support `minLength` or `default`, so those rules now live in the type hints and descriptions.

In [ ]:
type RegisterFunc = Callable[[Callable[..., Any]], Callable[..., Any]]

tools: dict[str, ToolsInfo] = {}


def register_func(name: str, schema: ToolParam, *, side_effects: bool = False) -> RegisterFunc:

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        if schema["name"] != name:
            raise TypeError(f"Schema name {schema['name']!r} != registed name {name!r}")
        input_schema = cast(
            dict[str, str | dict[str, str | dict[str, str]]], schema["input_schema"]
        )
        schema_params = set(input_schema.get("properties", {}))  # names Claude sees

        # .signature(func) reads a function's definition and returns a Signature object
        # that describes it. Its .parameters attribute is a read-only, ordered mapping:
        # each key is a parameter name, and each value is a Parameter object with the
        # details for that parameter.
        #
        # Calling set() on a mapping keeps only its keys, so this compares the two sets
        # of parameter names. If you rename timezone in the function but not in the schema,
        # the cell fails when you run it.
        func_signature = signature(func)

        func_params = set(func_signature.parameters)  # names Python accepts
        if schema_params != func_params:
            raise TypeError(
                f"{name}: schema params {sorted(schema_params)} "
                f"!= function params {sorted(func_params)}"
            )

        # Each Parameter has four fields:
        #   .name is the parameter's name.
        #   .default is its default value, or Parameter.empty if it has none
        #    (meaning the parameter is required).
        #   .annotation is the type hint. It shows up as a string like 'str' because of
        #    from __future__ import annotations.
        #   .kind says how the parameter can be passed: by position or keyword,
        #    keyword-only (after *), *args, or **kwargs.
        schema_required_params = set(input_schema.get("required", []))
        func_required_params = set(
            name for name, p in func_signature.parameters.items() if p.default is p.empty
        )

        if schema_required_params != func_required_params:
            raise TypeError(
                f"{name}: schema required params {sorted(schema_required_params)} "
                f"!= function required params {sorted(func_required_params)}"
            )

        validated = validate_call(func)
        tools[name] = ToolsInfo(validated, name=name, schema=schema, side_effects=side_effects)
        return validated

    return decorator

---
## CLIENT INITIALIZATION AND HELPER FUNCTIONS
---

In [ ]:
client, settings = get_client()


def add_user_message(
    messages: list[MessageParam],
    message: Message | list[ToolResultBlockParam] | str,
) -> None:
    user_message: MessageParam = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], message: Message | str) -> None:
    assistant_message: MessageParam = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    *,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
    tools: list[ToolParam] | None = None,
    tool_choice: ToolChoiceParam | None = None,
) -> Message:
    params: LLMParams = {
        "model": settings.claude_model,
        "max_tokens": settings.default_max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    if tool_choice:
        params["tool_choice"] = tool_choice

    # Console().print(params)
    return client.messages.create(**params)

---
## TOOL EXECUTION: errors become instructions, never crashes
---
- `_run_one` **never raises**. It returns `(ok, value)`. There are three kinds of failure: bad arguments (`ValidationError`), an expected failure (`ToolError`), and a bug in our code (any other exception, which is logged with its traceback).
- `_to_tool_content` serializes **once**. Before, `json.dumps(result)` was applied to a string that was already JSON, so Claude saw `"[{\"name\": ...}]"`.
- `run_tools` keeps what you already had right: one result per `tool_use`, all in one list, sent as one user message. It also adds a **circuit breaker**: if the same tool fails with the same arguments `MAX_SAME_FAILURES` times, the next identical call is not run, and Claude is told to change the arguments or stop. Better error text alone does not always break a retry loop, so the block is enforced in code.

In [ ]:
def text_from_message(message: Message) -> str:
    return "\n".join([block.text for block in message.content if isinstance(block, TextBlock)])


def _tools_from_message(message: Message) -> list[ToolUseBlock]:
    return [block for block in message.content if isinstance(block, ToolUseBlock)]


def _format_validation_error(exc: ValidationError) -> str:
    """One line Claude can act on: field path + reason, no docs URL."""
    parts: list[str] = []
    for err in exc.errors(include_url=False):
        loc = ".".join(str(p) for p in err["loc"]) or "input"
        parts.append(f"{loc}: {err['msg']}")
    return "; ".join(parts)


def _run_one(name: str, arguments: dict[str, Any]) -> tuple[bool, Any]:
    """Run one tool. Never raises. Returns (True, value) or (False, error message for Claude)."""
    info = tools.get(name)
    if info is None:
        return False, f"Unknown tool {name!r}. Available tools: {', '.join(tools)}"
    try:
        return True, info.func(**arguments)
    except ValidationError as exc:
        return False, f"Invalid arguments for {name}: {_format_validation_error(exc)}"
    except ToolError as exc:
        return False, f"{name} failed: {exc}"
    except Exception as exc:  # a bug in OUR code: keep the traceback in the logs
        logger.exception("Unexpected error in tool %s", name)
        return False, (
            f"{name} hit an internal error ({type(exc).__name__}: {exc}). "
            "Do not retry the same call; tell the user it failed."
        )


def _to_tool_content(value: Any) -> str:
    """Serialize exactly once: a str goes as-is, anything else becomes JSON."""
    return value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, default=str)


def _call_key(name: str, arguments: dict[str, Any]) -> str:
    return f"{name}:{json.dumps(arguments, sort_keys=True, default=str)}"


def run_tools(response: Message, failures: Counter[str]) -> list[ToolResultBlockParam]:
    """Answer EVERY tool_use block of this turn. The caller sends the list as ONE user message."""
    results: list[ToolResultBlockParam] = []
    for block in _tools_from_message(response):
        arguments = cast(dict[str, Any], block.input)
        key = _call_key(block.name, arguments)
        if failures[key] >= MAX_SAME_FAILURES:
            ok, value = (
                False,
                (
                    f"Not executed: this exact {block.name} call already failed "
                    f"{failures[key]} times. "
                    "Change the arguments, or stop and tell the user what went wrong."
                ),
            )
        else:
            started = time.perf_counter()
            ok, value = _run_one(block.name, arguments)
            logger.info(
                "tool=%s ok=%s %.1f ms", block.name, ok, (time.perf_counter() - started) * 1000
            )
            if not ok:
                failures[key] += 1
        results.append(
            ToolResultBlockParam(
                type="tool_result",
                tool_use_id=block.id,
                content=_to_tool_content(value),
                is_error=not ok,
            )
        )
    return results

---
## TOOLS AND SCHEMAS: ISO 8601 in and out, IANA timezones, DST-safe
---
- **One date format end to end.** Tools take ISO 8601 and return `{"iso", "human", "timezone"}`. Claude passes `iso` to the next tool and uses `human` in its answer. The old `input_format` / `date_format` strftime strings were a small language of their own that Claude could get wrong, and a mismatch caused the first failure in the loop.
- **IANA zone, not the server clock.** `datetime.now().astimezone()` gives a *fixed* offset (`EST`, −05:00). Add 177 days to it and you land in May, still at −05:00, which is 4 PM in New York instead of 3 PM. `ZoneInfo("America/New_York")` switches to EDT, −04:00. Abbreviations such as `EST` are rejected because `ZoneInfo("EST")` is a real zone pinned at −05:00 with no DST.
- **Calendar units vs elapsed units.** Days, weeks, months and years keep the clock time (3:00 PM stays 3:00 PM). Seconds, minutes and hours add real elapsed time, computed in UTC.
- **`set_reminder` is idempotent.** The same text at the same instant returns `already_exists` instead of creating a duplicate. That makes a retry harmless.

In [ ]:
Unit = Literal["seconds", "minutes", "hours", "days", "weeks", "months", "years"]
NonEmptyStr = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1)]
REMINDERS: dict[str, dict[str, str]] = {}  # stands in for a database table


def _zone(name: str) -> ZoneInfo:
    if "/" not in name and name != "UTC":
        raise ToolError(
            f"Timezone {name!r} is an abbreviation or legacy name. Use an IANA region name "
            "such as 'America/New_York so daylight saving time is handled."
        )
    try:
        return ZoneInfo(name)
    except (ZoneInfoNotFoundError, ValueError) as exc:
        raise ToolError(
            f"Unknown timezone {name!r}. Use an IANA name such as 'America/New_York'."
        ) from exc


def _parse_iso(value: str, field: str) -> datetime:
    try:
        return datetime.fromisoformat(value)
    except ValueError as exc:
        raise ToolError(
            f"{field}={value!r} is not ISO 8601. Send e.g. '2026-11-26T15:00:00' "
            "or '2026-11-26T15:00:00-05_00'."
        ) from exc


def _describe(dt: datetime, tz_name: str) -> dict[str, str]:
    """Machine value for the next tool + human value for the final answer."""
    return {
        "iso": dt.isoformat(timespec="seconds"),
        "human": dt.strftime("%A, %B, %d, %Y %I:%M %p %Z"),
        "timezone": tz_name,
    }


def _add_months(dt: datetime, months: int) -> datetime:
    total = dt.month - 1 + months
    year, month = dt.year + total // 12, total % 12 + 1
    last_day = (datetime(year + month // 12, month % 12 + 1, 1) + timedelta(days=1)).day
    return dt.replace(year=year, month=month, day=min(dt.day, last_day))


get_current_datetime_schema: ToolParam = {
    "name": "get_current_datetime",
    "description": f"""Returns the current date and time in the user's timezone. Use it whenever an
    answer depends on the present moment: today's date, the time now, or the starting point for
    "N days from now". It only reads the clock; use add_duration_to_datetime for date math.
    Returns JSON like {{"iso": "2026-11-26T17:09:05-05:00", "human": "Thursday, November 26, 2026
    05:09 PM EST", "timezone": "{DEFAULT_TZ}"}}. Pass the "iso" value to other tools.""",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "timezone": {
                "type": "string",
                "description": f"IANA timezone name, e.g. 'America/Chicago'. Omit it to use the "
                f"user's timezone ({DEFAULT_TZ}). Abbreviations like 'EST' are rejected.",
            }
        },
        "required": [],
        "additionalProperties": False,
    },
    "input_examples": [{}, {"timezone": "America/Chicago"}],
}


@register_func("get_current_datetime", get_current_datetime_schema)
def get_current_datetime(timezone: str = DEFAULT_TZ) -> dict[str, str]:
    return _describe(datetime.now(_zone(timezone)), timezone)


add_duration_to_datetime_schema: ToolParam = {
    "name": "add_duration_to_datetime",
    "description": """Adds a duration to a datetime and returns the result in the given IANA
    timezone, with daylight saving time handled. Days, weeks, months and years keep the same
    clock time (3:00 PM stays 3:00 PM across a DST change); seconds, minutes and hours add exact
    elapsed time. To land on a specific clock time, pass the start date with that time,
    e.g. '2026-11-26T15:00:00'. Months and years clamp to the last valid day
    (Jan 31 + 1 month = Feb 28). Returns JSON {"iso", "human", "timezone"}.
    Pass "iso" to set_reminder.""",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "Start datetime in ISO 8601, either '2026-11-26T15:00:00' (read in "
                "`timezone`) or with an offset, '2026-11-26T15:00:00-05:00'. "
                "A date alone means midnight.",
            },
            "duration": {
                "type": "integer",
                "description": "Whole number of units to add. Negative values go back in time.",
            },
            "unit": {
                "type": "string",
                "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
                "description": "Unit of the duration. Defaults to 'days' when omitted.",
            },
            "timezone": {
                "type": "string",
                "description": "IANA timezone for the calculation and the result. "
                "Defaults to the user's timezone.",
            },
        },
        "required": ["datetime_str", "duration"],
        "additionalProperties": False,
    },
}


@register_func("add_duration_to_datetime", add_duration_to_datetime_schema)
def add_duration_to_datetime(
    datetime_str: str, duration: int, unit: Unit = "days", timezone: str = DEFAULT_TZ
) -> dict[str, str]:
    zone = _zone(timezone)
    start = _parse_iso(datetime_str, "datetime_str")
    start = start.replace(tzinfo=zone) if start.tzinfo is None else start.astimezone(zone)

    match unit:
        case "seconds" | "minutes" | "hours":
            # Elapsed time: add in UTC so a DST jump can't add or remove an hour.
            result = (start.astimezone(UTC) + timedelta(**{unit: duration})).astimezone(zone)
        case "days" | "weeks":
            # Calendar time: aware datetime + timedelta keeps the wall clock, and ZoneInfo
            # picks the right offset for the new date (EST -05:00 -> EDT -04:00).
            result = start + timedelta(**{unit: duration})
        case "months":
            result = _add_months(start, duration)
        case "years":
            result = _add_months(start, duration * 12)
    return _describe(result, timezone)


set_reminder_schema: ToolParam = {
    "name": "set_reminder",
    "description": """Creates a reminder that notifies the user at the given time. The timestamp
    must be in the future and must include a UTC offset: pass the "iso" value returned by
    add_duration_to_datetime. Safe to retry: the same content and time return the existing reminder
    (status "already_exists") instead of creating a duplicate.""",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "Reminder text shown to the user, e.g. 'Doctor appointment'.",
            },
            "timestamp": {
                "type": "string",
                "description": "When to notify, ISO 8601 with offset, "
                "e.g. '2027-05-22T15:00:00-04:00'.",
            },
        },
        "required": ["content", "timestamp"],
        "additionalProperties": False,
    },
}


@register_func("set_reminder", set_reminder_schema, side_effects=True)
def set_reminder(content: NonEmptyStr, timestamp: str) -> dict[str, str]:
    when = _parse_iso(timestamp, "timestamp")
    if when.tzinfo is None:
        raise ToolError(
            f"timestamp={timestamp!r} has not UTC offset, so the time is ambiguous. Send it with "
            "an offset, e.g. '2027-05-22T15:00:00-04:00' "
            "(the 'iso' value from add_duration_to_datetime)."
        )
    if when <= datetime.now(UTC):
        raise ToolError(f"timestamp={timestamp!r} is in the past. Reminders must be in the future.")

    # Idempotency key: same text + same instant -> same key -> a retry can't create a duplicate.
    instant = when.astimezone(UTC).isoformat()
    reminder_id = hashlib.sha256(f"{content.casefold()}|{instant}".encode()).hexdigest()[:12]
    if reminder_id in REMINDERS:
        return {"status": "already_exists", **REMINDERS[reminder_id]}
    REMINDERS[reminder_id] = {
        "reminder_id": reminder_id,
        "content": content,
        "timestamp": when.isoformat(timespec="seconds"),
    }
    return {"status": "created", **REMINDERS[reminder_id]}

---
## BATCH TOOL (optional, fixed)
---
Keep it for the course exercise, then compare runs with it on and off. Changes from the original:
- Each item is validated **on its own**, so one bad item can't sink the others.
- Results are reported **per item** (`ok`, `result` or `error`). The batch is `is_error=True` only if **every** item failed, because Claude must not retry items that already ran.
- The description tells Claude not to batch dependent calls. "Now", then "+177 days", then "set the reminder" is a chain: each step needs the previous output, so it takes three turns whatever tool you give it.

In [ ]:
batch_tool_schema: ToolParam = {
    "name": "batch_tool",
    "description": """Runs several INDEPENDENT tool calls in one step. Do not batch a call that
    needs another call's output; make that call in the next turn instead. Returns JSON with one
    entry per invocation, in order: {"index", "name", "ok", "result" or "error"}. Entries are
    independent: a failed entry does not undo or block the others, so only retry the entries
    that failed.""",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to run.",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "Name of the tool to run."},
                        "arguments": {
                            "type": "string",
                            "description": "The tool's arguments as a JSON-encoded object string, "
                            'e.g. \'{"timezone": "America/Chicago"}\'.',
                        },
                    },
                    "required": ["name", "arguments"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["invocations"],
        "additionalProperties": False,
    },
}


def batch_tool(invocations: list[dict[str, Any]]) -> dict[str, Any]:
    entries: list[dict[str, Any]] = []
    for index, raw in enumerate(invocations):
        try:
            inv = Invocation.model_validate(raw)
        except ValidationError as exc:
            entries.append(
                {
                    "index": index,
                    "name": raw.get("name"),
                    "ok": False,
                    "error": f"Invalid invocation: {_format_validation_error(exc)}",
                }
            )
            continue
        if inv.name == "batch_tool":
            ok, value = False, "batch_tool cannot call itself."
        else:
            ok, value = _run_one(inv.name, inv.arguments)
        entries.append(
            {"index": index, "name": inv.name, "ok": ok, ("result" if ok else "erroe"): value}
        )

    succeeded = sum(1 for e in entries if e["ok"])
    summary: dict[str, Any] = {
        "succeeded": succeeded,
        "failed": len(entries) - succeeded,
        "results": entries,
    }
    if entries and succeeded == 0:
        raise ToolError(
            json.dumps(summary, ensure_ascii=False, default=str)
        )  # all failed -> is_error
    return summary


if ENABLE_BATCH_TOOL:
    register_func("batch_tool", batch_tool_schema)(batch_tool)

---
## AGENT LOOP: bounded, forced only on turn 1, explicit stop reasons
---
- **`first_tool_choice` applies to turn 1 only.** A forced choice (`{"type": "tool", ...}` or `{"type": "any"}`) means "you must call a tool now". Sent on every turn, Claude can never end. Forcing also suppresses the text Claude would write before the call, and some newer models (Opus 5.5, Sonnet 5.5) don't accept `any`/`tool` at all. `claude-haiku-4-5`, your default, does.
- **`match response.stop_reason`.** `tool_use` continues the loop. `end_turn` and `stop_sequence` finish. `max_tokens` raises, because a tool call may have been cut off mid-JSON. `refusal` returns. Anything else raises instead of looping blindly.
- **Budget exhausted?** One last call with `tool_choice={"type": "none"}`, so you get a summary of what was done instead of a hang.
- **`RunResult`** reports turns, tool calls and tokens, the minimum you need to watch cost and loop health.

In [ ]:
@dataclass(slots=True, kw_only=True)
class RunResult:
    final: Message
    turns: int
    tool_calls: int
    input_tokens: int
    output_tokens: int
    stopped_early: bool = False


class AgentLoopError(RuntimeError):
    """The loop could not finish safely (e.g. a tool call was cut off by max_tokens)."""


def run_conversation(
    messages: list[MessageParam],
    *,
    max_turns: int = MAX_TURNS,
    first_tool_choice: ToolChoiceParam | None = None,
    **chat_settings: Any,
) -> RunResult:
    failures: Counter[str] = Counter()
    tool_calls = input_tokens = output_tokens = 0

    for turn in range(1, max_turns + 1):
        tool_choice = first_tool_choice if turn == 1 else None
        response = chat(messages, tool_choice=tool_choice, **chat_settings)
        add_assistant_message(messages, response)
        input_tokens += response.usage.input_tokens
        output_tokens += response.usage.output_tokens

        if text := text_from_message(response):
            console.print(Markdown(text, code_theme="monokai"))

        match response.stop_reason:
            case "tool_use":
                results = run_tools(response, failures)
                tool_calls += len(results)
                add_user_message(messages, results)
            case "end_turn" | "stop_sequence" | "refusal":
                return RunResult(
                    final=response,
                    turns=turn,
                    tool_calls=tool_calls,
                    input_tokens=input_tokens,
                    output_tokens=output_tokens,
                )
            case "max_tokens":
                raise AgentLoopError(
                    "Response hit max_tokens, so a tool call may be cut off. "
                    "Raise default_max_tokens and run again."
                )
            case other:
                raise AgentLoopError(f"Unhandled stop_reason: {other!r}")

    logger.warning("Hit max_turns=%d; asking for a final answer with tools disabled.", max_turns)
    response = chat(messages, tool_choice={"type": "none"}, **chat_settings)
    add_assistant_message(messages, response)
    input_tokens += response.usage.input_tokens
    output_tokens += response.usage.output_tokens
    if text := text_from_message(response):
        console.print(Markdown(text, code_theme="monokai"))
    return RunResult(
        final=response,
        turns=max_turns + 1,
        tool_calls=tool_calls,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        stopped_early=True,
    )

---
## RUN THE CONVERSATION
---
Expect **4 model calls**: read the clock, add 177 days at 3 PM, set the reminder, then the final answer. The first three each need the previous output, so they can't run in parallel. With `ENABLE_BATCH_TOOL = True` you should see the same count, which shows the batch tool doesn't help a dependent chain.

In [ ]:
system_prompt: str = """
You are an appointment scheduler specialist. You communicate the steps you are
following to complete the set up, and you communicate any delays you may have.
At the end you provide the result in a professional and friendly way.
When one tool needs another tool's output, call them in separate turns; only call
tools in parallel when they are independent of each other. Pass the "iso" values
that tools return straight into the next tool.
"""

list_tools: list[ToolParam] = [info.schema for info in tools.values()]
messages: list[MessageParam] = []
user_prompt: str = """
Set a reminder for my doctors appt. Its 177 days from now, at 3 pm my time zone.
"""
add_user_message(messages, user_prompt)

first_tool_choice: ToolChoiceParam = {
    "name": "batch_tool",
    "type": "tool",
}

result = run_conversation(
    messages,
    first_tool_choice=None,
    system=system_prompt,
    tools=list_tools,
)
# To force the FIRST step only (e.g. always read the clock first):
# result = run_conversation(messages, system=system_prompt, tools=list_tools,
#                           first_tool_choice={"type": "tool", "name": "get_current_datetime"})

# To inspect the raw dicts while debugging, use console.print_json(data=r) or console.print(r).
# Both print them with colors and indentation, and Markdown rendering would hide the structure.
# console.print(tools)
console.print(result)
# console.print(REMINDERS)
console.print(messages)  # full transcript for debugging